# Power Grid Analysis & IR-Drop Optimization (7nm FinFET) Tutorial

This interactive tutorial demonstrates how to use the `pdn_analysis` toolkit to:
1. Size metal power stripes and rails analytically based on **Electromigration (EM)** and **Static IR Drop** limits.
2. Calculate **Decap sizing** and analyze the **Frequency-Domain PDN Impedance spectrum $Z(f)$** against $Z_{target}$.
3. Solve 2D/multi-layer power mesh networks using **Modified Nodal Analysis (MNA)**.
4. Explore the **Pareto Trade-off Frontier** (Metal Routing Blockage vs. Peak IR Drop).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from core.technology_specs import TechnologyConfig
from core.analytical_sizing import PDNAnalyticalSizer
from core.impedance_analyzer import PDNImpedanceAnalyzer
from core.mna_mesh_solver import MNAPowerGridSolver, GridConfig, PowerHotspot
from core.optimizer import PDNOptimizer

tech = TechnologyConfig()
sizer = PDNAnalyticalSizer(tech)
print(f"Technology: {tech.node_name}")
print(f"Nominal VDD: {tech.nominal_vdd} V")
print(f"Static IR Budget: {tech.static_ir_budget_volts*1e3:.1f} mV")
print(f"Dynamic IR Budget: {tech.dynamic_ir_budget_volts*1e3:.1f} mV")

## 1. Analytical Sizing for Power Stripes & Rails
We calculate the required width of power stripes on M1, M4, and M8 to meet EM and IR drop constraints.

In [ ]:
m8_res = sizer.size_power_stripe(
    layer_name="M8",
    pitch_um=30.0,
    span_length_um=120.0,
    power_density_w_per_mm2=1.2,
    target_ir_drop_v=0.012,
    feed_type="both_ends"
)
print(f"M8 Sizing Result:")
print(f"  - Width for IR: {m8_res.w_min_ir_um:.2f} µm")
print(f"  - Width for EM: {m8_res.w_min_em_um:.2f} µm")
print(f"  - Recommended Width: {m8_res.recommended_width_um:.2f} µm")
print(f"  - Metal Blockage: {m8_res.metal_utilization_pct:.1f}%")
print(f"  - Actual Static IR Drop: {m8_res.actual_static_ir_drop_mv:.2f} mV")
print(f"  - EM Safety Margin: {m8_res.em_safety_margin:.2f}x")

## 2. PDN Frequency-Domain Impedance & Target Impedance $Z(f)$
Larry Smith's target impedance: $Z_{target} = \frac{V_{DD} \cdot \text{ripple}}{\Delta I}$. Anti-resonance occurs at $f_{res} = \frac{1}{2\pi\sqrt{L_{pkg} C_{decap}}}$.

In [ ]:
analyzer = PDNImpedanceAnalyzer()
z_target = analyzer.calculate_target_impedance(vdd=0.80, allowed_ripple_pct=0.05, transient_step_current_a=2.0)
print(f"Target Impedance: {z_target*1e3:.2f} mOhm")

# Compute frequency spectrum for 2 different decap values (5 nF vs 20 nF)
spec_low = analyzer.calculate_impedance_spectrum(r_pkg_ohm=0.005, l_pkg_h=20e-12, c_decap_f=5e-9, target_z_ohm=z_target)
spec_high = analyzer.calculate_impedance_spectrum(r_pkg_ohm=0.005, l_pkg_h=20e-12, c_decap_f=20e-9, target_z_ohm=z_target)

plt.figure(figsize=(9, 5))
plt.loglog(spec_low.frequencies_hz, spec_low.z_magnitude_ohm * 1e3, label=f"Low Decap (5 nF) - Peak={spec_low.peak_impedance_ohm*1e3:.1f} mΩ", color="red")
plt.loglog(spec_high.frequencies_hz, spec_high.z_magnitude_ohm * 1e3, label=f"High Decap (20 nF) - Peak={spec_high.peak_impedance_ohm*1e3:.1f} mΩ", color="blue")
plt.axhline(z_target * 1e3, color="green", linestyle="--", label=f"Z_target ({z_target*1e3:.1f} mΩ)")
plt.title("PDN Impedance Spectrum vs Target Impedance", fontweight="bold")
plt.xlabel("Frequency (Hz)")
plt.ylabel("|Z(f)| (mOhm)")
plt.grid(True, which="both", ls=":", alpha=0.6)
plt.legend()
plt.show()

## 3. Modified Nodal Analysis (MNA) 2D Numerical Simulation
We model a $600\,\mu\text{m} \times 600\,\mu\text{m}$ die with Core ALU and NPU hotspots.

In [ ]:
cfg = GridConfig(
    chip_width_um=600.0,
    chip_height_um=600.0,
    h_width_um=3.5,
    h_pitch_um=30.0,
    v_width_um=3.0,
    v_pitch_um=25.0,
    bump_pitch_um=120.0,
    background_power_w_per_mm2=0.7,
    hotspots=[
        PowerHotspot("Core_ALU", 150.0, 320.0, 150.0, 320.0, power_density_w_per_mm2=3.2),
        PowerHotspot("NPU", 380.0, 520.0, 360.0, 500.0, power_density_w_per_mm2=2.8),
    ],
    uniform_decap_nf_per_mm2=15.0
)

solver = MNAPowerGridSolver(tech, cfg)
static_res = solver.solve_static()

print(f"Peak Static IR Drop: {static_res.max_ir_drop_mv:.2f} mV")
print(f"Average Static IR Drop: {static_res.avg_ir_drop_mv:.2f} mV")
print(f"Max Current Density: {static_res.max_current_density_ma_per_um2:.2f} mA/µm²")
print(f"EM Violations: {static_res.em_violation_count}")

plt.figure(figsize=(7, 6))
X, Y = np.meshgrid(static_res.grid_x, static_res.grid_y)
plt.contourf(X, Y, static_res.ir_drop_map, levels=25, cmap="inferno")
plt.colorbar(label="Static IR Drop (mV)")
plt.title(f"Static IR Drop Heatmap (Peak = {static_res.max_ir_drop_mv:.2f} mV)", fontweight="bold")
plt.xlabel("X (µm)")
plt.ylabel("Y (µm)")
plt.show()